# FNN Pure Python Windows EXE 패키지 생성

이 노트북은 `1 → 8 → 8 → 8 → 1` PyTorch FNN 모델을 대상으로 다음 작업을 수행합니다.

- `/content/Basic_example_copy/results/fnn_model.pt` 로드
- Linear 계층의 weight와 bias를 일반 Python `list`로 변환
- PyTorch 출력과 Pure Python 출력의 백투백 검증
- `tkinter` 기반 Pure Python GUI 코드 생성
- Windows용 `build_exe.bat` 생성
- `FNN_PurePython_EXE_Package.zip` 자동 다운로드

최종 EXE는 PyTorch, PyQt6, NumPy, joblib 및 별도 `.pt` 파일을 사용하지 않습니다.

> Google Colab에서는 Windows EXE 자체를 만들지 않습니다. Colab이 내려주는 ZIP을 Windows에서 압축 해제한 후 `build_exe.bat`을 실행하면 `exe_folder/fnn_model_pure_python.exe`가 생성됩니다.


## 실행 전 확인

아래 실행 셀 상단의 경로를 필요에 따라 수정하세요.

```python
REPO_ROOT = Path("/content/Basic_example_copy").resolve()
RESULT_DIR = REPO_ROOT / "results"
PT_PATH = RESULT_DIR / "fnn_model.pt"
```

`fnn_model.pt`는 다음 모델의 `state_dict` 또는 `model_state_dict`를 포함한 체크포인트여야 합니다.

```text
1 → 8 → 8 → 8 → 1
활성화 함수: ReLU
```


In [ ]:
"""Colab에서 순수 Python 기반 Windows EXE 빌드 패키지를 생성합니다.

이 파일은 Colab의 마지막 셀에 붙여 넣어 실행할 수 있습니다.
Colab에서는 PyTorch 모델의 가중치를 일반 Python list로 변환합니다.
최종 Windows 프로그램은 tkinter와 Python 기본 연산만 사용합니다.
"""

from pathlib import Path
import shutil
import subprocess

import torch
import torch.nn as nn
from google.colab import files


# =========================================================
# 사용자 설정
# =========================================================
REPO_ROOT = Path("/content/Basic_example_copy").resolve()
RESULT_DIR = REPO_ROOT / "results"
PT_PATH = RESULT_DIR / "fnn_model.pt"

PACKAGE_DIR = RESULT_DIR / "FNN_PurePython_EXE_Package"
ZIP_BASE = RESULT_DIR / "FNN_PurePython_EXE_Package"

X_MEAN = 20.0
X_SCALE = 11.5759087
Y_MEAN = 5.44184351
Y_SCALE = 3.66043448


# =========================================================
# GitHub 저장소와 모델 확인
# =========================================================
if not REPO_ROOT.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "-q",
            "https://github.com/Jaehoon-Shim/Basic_example.git",
            str(REPO_ROOT),
        ],
        check=True,
    )

RESULT_DIR.mkdir(parents=True, exist_ok=True)

if not PT_PATH.is_file():
    available_models = sorted(RESULT_DIR.glob("*.pt"))
    model_list = "\n".join(str(path) for path in available_models)

    raise FileNotFoundError(
        "fnn_model.pt를 찾을 수 없습니다.\n\n"
        f"확인 경로: {PT_PATH}\n\n"
        "results 폴더에서 발견된 PT 파일:\n"
        f"{model_list or '없음'}"
    )


# =========================================================
# 학습 때 사용한 FNN 구조
# Architecture: 1 -> 8 -> 8 -> 8 -> 1
# =========================================================
class FNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(1, 8),
            nn.ReLU(),
            nn.Linear(8, 8),
            nn.ReLU(),
            nn.Linear(8, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
        )

    def forward(self, x):
        return self.model(x)


# =========================================================
# PT 모델 로드
# =========================================================
checkpoint = torch.load(
    PT_PATH,
    map_location="cpu",
    weights_only=True,
)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    state_dict = checkpoint["model_state_dict"]
else:
    state_dict = checkpoint

# DataParallel로 저장된 경우 module. 접두어 제거
if state_dict and all(str(key).startswith("module.") for key in state_dict):
    state_dict = {
        str(key).removeprefix("module."): value
        for key, value in state_dict.items()
    }

model_cpu = FNN()
model_cpu.load_state_dict(state_dict)
model_cpu.to("cpu")
model_cpu.eval()


# =========================================================
# Linear 파라미터를 일반 Python list로 변환
# =========================================================
linear_layers = [
    layer
    for layer in model_cpu.model
    if isinstance(layer, nn.Linear)
]

embedded_layers = [
    {
        "weight": layer.weight.detach().cpu().tolist(),
        "bias": layer.bias.detach().cpu().tolist(),
    }
    for layer in linear_layers
]

expected_shapes = [
    (8, 1),
    (8, 8),
    (8, 8),
    (1, 8),
]

actual_shapes = [
    tuple(layer.weight.shape)
    for layer in linear_layers
]

if actual_shapes != expected_shapes:
    raise ValueError(
        "모델 구조가 1-8-8-8-1과 다릅니다.\n"
        f"기대 Weight 형상: {expected_shapes}\n"
        f"실제 Weight 형상: {actual_shapes}"
    )


# =========================================================
# PyTorch ↔ 순수 Python 백투백 검증
# =========================================================
def dense(values, weight, bias):
    return [
        sum(w * x for w, x in zip(row, values)) + b
        for row, b in zip(weight, bias)
    ]


def predict_pure_python(x_value):
    values = [(float(x_value) - X_MEAN) / X_SCALE]

    for index, layer in enumerate(embedded_layers):
        values = dense(
            values,
            layer["weight"],
            layer["bias"],
        )

        if index < len(embedded_layers) - 1:
            values = [max(0.0, value) for value in values]

    return values[0] * Y_SCALE + Y_MEAN


def predict_pytorch(x_value):
    x_scaled = (float(x_value) - X_MEAN) / X_SCALE
    x_tensor = torch.tensor([[x_scaled]], dtype=torch.float32)

    with torch.inference_mode():
        y_scaled = model_cpu(x_tensor).item()

    return y_scaled * Y_SCALE + Y_MEAN


verification_inputs = [
    -10.0,
    0.0,
    5.0,
    10.0,
    20.0,
    30.0,
    40.0,
    50.0,
]

verification_rows = []

for x_value in verification_inputs:
    pytorch_output = predict_pytorch(x_value)
    pure_output = predict_pure_python(x_value)
    absolute_error = abs(pytorch_output - pure_output)

    verification_rows.append(
        (
            x_value,
            pytorch_output,
            pure_output,
            absolute_error,
        )
    )

max_absolute_error = max(row[3] for row in verification_rows)

print("PyTorch ↔ 순수 Python 백투백 검증")
print("-" * 72)
print(f"{'입력':>10} {'PyTorch':>18} {'Pure Python':>18} {'절대오차':>18}")

for row in verification_rows:
    print(
        f"{row[0]:10.4f} "
        f"{row[1]:18.10f} "
        f"{row[2]:18.10f} "
        f"{row[3]:18.10e}"
    )

print("-" * 72)
print(f"최대 절대오차: {max_absolute_error:.10e}")

if max_absolute_error > 1.0e-5:
    raise RuntimeError(
        "순수 Python 변환 결과가 PyTorch 결과와 허용오차를 초과했습니다.\n"
        f"최대 절대오차: {max_absolute_error}"
    )


# =========================================================
# 순수 Python 실행 프로그램 템플릿
#
# 최종 앱의 외부 의존성:
#   없음
#
# 사용 모듈:
#   tkinter, math 등의 Python 표준 라이브러리만 사용
# =========================================================
APP_TEMPLATE = r'''import tkinter as tk
from tkinter import messagebox, ttk


# =========================================================
# Embedded FNN parameters
# Architecture: 1 -> 8 -> 8 -> 8 -> 1
# =========================================================
LAYERS = __LAYERS__


# =========================================================
# StandardScaler parameters
# =========================================================
X_MEAN = __X_MEAN__
X_SCALE = __X_SCALE__
Y_MEAN = __Y_MEAN__
Y_SCALE = __Y_SCALE__


# =========================================================
# Pure Python FNN inference
# =========================================================
def dense(values, weight, bias):
    """Fully connected layer: output = weight * input + bias."""
    return [
        sum(w * x for w, x in zip(row, values)) + b
        for row, b in zip(weight, bias)
    ]


def relu(values):
    """ReLU activation."""
    return [max(0.0, value) for value in values]


def predict(x_value):
    """Standardization -> FNN -> inverse standardization."""
    values = [(float(x_value) - X_MEAN) / X_SCALE]

    for index, layer in enumerate(LAYERS):
        values = dense(
            values,
            layer["weight"],
            layer["bias"],
        )

        if index < len(LAYERS) - 1:
            values = relu(values)

    return values[0] * Y_SCALE + Y_MEAN


# =========================================================
# tkinter GUI
# =========================================================
class App(tk.Tk):
    def __init__(self):
        super().__init__()

        self.title("FNN Model Inference - Pure Python")
        self.geometry("520x350")
        self.resizable(False, False)
        self.configure(padx=30, pady=25)

        style = ttk.Style(self)

        style.configure(
            "Title.TLabel",
            font=("맑은 고딕", 18, "bold"),
            foreground="#0B2A53",
        )

        style.configure(
            "Input.TLabel",
            font=("맑은 고딕", 11, "bold"),
        )

        style.configure(
            "Result.TLabel",
            font=("맑은 고딕", 16, "bold"),
            foreground="#174D89",
        )

        ttk.Label(
            self,
            text="FNN 회귀모델 추론",
            style="Title.TLabel",
        ).pack(pady=(0, 24))

        input_frame = ttk.Frame(self)
        input_frame.pack(fill="x")

        ttk.Label(
            input_frame,
            text="입력값 x",
            style="Input.TLabel",
        ).pack(side="left")

        self.x_entry = ttk.Entry(
            input_frame,
            width=27,
            font=("맑은 고딕", 11),
        )
        self.x_entry.pack(side="right")

        ttk.Button(
            self,
            text="예측 실행",
            command=self.run_inference,
        ).pack(fill="x", pady=22)

        result_frame = ttk.LabelFrame(
            self,
            text="추론 결과",
            padding=14,
        )
        result_frame.pack(fill="x", ipady=15)

        ttk.Label(
            result_frame,
            text="예측값 y",
            style="Input.TLabel",
        ).pack(side="left", pady=10)

        self.result = tk.StringVar(value="-")

        ttk.Label(
            result_frame,
            textvariable=self.result,
            style="Result.TLabel",
        ).pack(side="right", pady=10)

        developer_label = ttk.Label(
            self,
            text="Developed by Jaehoon Shim (SEED Lab)",
            foreground="#888888",
        )
        developer_label.pack(anchor="e", pady=(18, 0))

        self.x_entry.bind(
            "<Return>",
            lambda _event: self.run_inference(),
        )
        self.x_entry.focus_set()

    def run_inference(self):
        try:
            x_value = float(self.x_entry.get().strip())
            y_value = predict(x_value)
            self.result.set(f"{y_value:.8g}")

        except ValueError:
            messagebox.showwarning(
                "입력 오류",
                "x 값에 숫자를 입력해 주세요.",
            )

        except Exception as error:
            messagebox.showerror(
                "실행 오류",
                str(error),
            )


if __name__ == "__main__":
    App().mainloop()
'''


# 가중치와 StandardScaler 상수를 소스코드에 삽입
app_code = APP_TEMPLATE.replace(
    "__LAYERS__",
    repr(embedded_layers),
)
app_code = app_code.replace("__X_MEAN__", repr(float(X_MEAN)))
app_code = app_code.replace("__X_SCALE__", repr(float(X_SCALE)))
app_code = app_code.replace("__Y_MEAN__", repr(float(Y_MEAN)))
app_code = app_code.replace("__Y_SCALE__", repr(float(Y_SCALE)))


# =========================================================
# Windows EXE 빌드 BAT
#
# PyTorch, PyQt6, NumPy, joblib 설치 불필요
# Python과 PyInstaller만 사용
# =========================================================
BUILD_BAT = r'''@echo off
setlocal
chcp 65001 > nul
cd /d "%~dp0"

echo ========================================
echo Pure Python FNN Windows EXE Build
echo ========================================
echo.

if not exist "FNN_Inference_PurePython.py" (
    echo [오류] FNN_Inference_PurePython.py 파일이 없습니다.
    pause
    exit /b 1
)

REM ----------------------------------------
REM 출력 폴더를 명령 실행 전에 생성
REM ----------------------------------------
if not exist "exe_folder" (
    mkdir "exe_folder"
)

REM ----------------------------------------
REM Python 실행 명령 확인
REM ----------------------------------------
set "PYTHON_CMD="

where py >nul 2>nul
if not errorlevel 1 set "PYTHON_CMD=py -3"

if not defined PYTHON_CMD (
    where python >nul 2>nul
    if not errorlevel 1 set "PYTHON_CMD=python"
)

REM ----------------------------------------
REM Python이 없으면 자동 설치
REM ----------------------------------------
if not defined PYTHON_CMD (
    echo Python이 없어 Python 3.12 설치를 시작합니다.

    where winget >nul 2>nul
    if errorlevel 1 (
        echo [오류] winget을 찾지 못했습니다.
        echo Microsoft Store 또는 python.org에서
        echo Python 3.12를 설치한 후 다시 실행하세요.
        pause
        exit /b 1
    )

    winget install ^
      --id Python.Python.3.12 ^
      -e ^
      --scope user ^
      --silent ^
      --accept-package-agreements ^
      --accept-source-agreements

    if errorlevel 1 (
        echo [오류] Python 자동 설치에 실패했습니다.
        pause
        exit /b 1
    )

    echo.
    echo Python 설치가 완료되었습니다.
    echo 현재 창에서 Python이 인식되지 않으면
    echo 창을 닫고 build_exe.bat을 다시 실행하세요.
    echo.

    set "PYTHON_CMD=py -3"
)

echo Python 환경을 확인합니다.
%PYTHON_CMD% --version

if errorlevel 1 (
    echo [오류] Python을 실행할 수 없습니다.
    echo 이 창을 닫은 후 build_exe.bat을 다시 실행하세요.
    pause
    exit /b 1
)

REM tkinter는 Windows용 기본 Python에 포함됩니다.
%PYTHON_CMD% -c "import tkinter" >nul 2>nul

if errorlevel 1 (
    echo [오류] tkinter를 사용할 수 없습니다.
    echo python.org의 Windows용 표준 Python을 설치해 주세요.
    pause
    exit /b 1
)

echo.
echo PyInstaller를 설치합니다.
%PYTHON_CMD% -m ensurepip --upgrade
%PYTHON_CMD% -m pip install --upgrade pip pyinstaller

if errorlevel 1 (
    echo [오류] PyInstaller 설치에 실패했습니다.
    pause
    exit /b 1
)

echo.
echo fnn_model_pure_python.exe 생성을 시작합니다.
echo.

%PYTHON_CMD% -m PyInstaller ^
  --noconfirm ^
  --clean ^
  --onefile ^
  --windowed ^
  --name fnn_model_pure_python ^
  --distpath "exe_folder" ^
  FNN_Inference_PurePython.py

if errorlevel 1 (
    echo.
    echo [오류] EXE 생성에 실패했습니다.
    pause
    exit /b 1
)

echo.
echo ========================================
echo EXE 생성 완료
echo.
echo 결과:
echo exe_folder\fnn_model_pure_python.exe
echo.
echo PyTorch와 별도의 PT 모델 파일이 필요하지 않습니다.
echo 최종 실행 PC에도 Python 설치가 필요하지 않습니다.
echo ========================================
pause
'''


# =========================================================
# 패키지 생성
# =========================================================
if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)

PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

app_path = PACKAGE_DIR / "FNN_Inference_PurePython.py"
bat_path = PACKAGE_DIR / "build_exe.bat"

app_path.write_text(app_code, encoding="utf-8")

with open(
    bat_path,
    "w",
    encoding="utf-8",
    newline="\r\n",
) as file:
    file.write(BUILD_BAT)


# =========================================================
# ZIP 생성 및 다운로드
# =========================================================
zip_path = Path(
    shutil.make_archive(
        str(ZIP_BASE),
        "zip",
        PACKAGE_DIR,
    )
)

print()
print("순수 Python 실행 프로그램:", app_path)
print("Windows 빌드 BAT:", bat_path)
print("Windows 빌드 패키지:", zip_path)
print()
print("Windows 사용 방법:")
print("1. ZIP 압축 해제")
print("2. build_exe.bat 실행")
print("3. exe_folder/fnn_model_pure_python.exe 확인")
print()
print("최종 앱 외부 라이브러리: 없음")
print("최종 앱 별도 모델 파일: 없음")

files.download(str(zip_path))
